[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/10_Feedback_Control.ipynb)

# Notebook 10 — Feedback Control

**Companion to Chapter 10**

This notebook stabilizes the locally unstable diver model with proportional–derivative feedback and checks the design through poles, trajectories, disturbances, and actuator limits.

## Learning objectives

By the end of this notebook, you should be able to:

- design PD gains from desired closed-loop poles;
- verify stabilization in state and time-domain simulations;
- predict constant-disturbance offset;
- explain how force saturation changes the nominal response;

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})

m = 85.0                 # kg
rho = 1025.0             # kg/m^3
g = 9.80665              # m/s^2
z_star = 20.0            # m, positive downward
V_g0 = 8.0e-3            # m^3 at the surface
p_atm = 101325.0          # Pa
p_star = p_atm + rho*g*z_star
V_g_star = V_g0*p_atm/p_star
k_B = -rho**2*g**2*V_g_star/p_star
a = k_B/m

A = np.array([[0.0, -1.0], [a, 0.0]])
B = np.array([[0.0], [1.0/m]])         # positive input force is upward
C = np.array([[1.0, 0.0]])
D = np.zeros((1, 1))

print(f"Local buoyancy slope k_B = {k_B:.4f} N/m")
print(f"Plant coefficient a = {a:.6f} s^-2")

## 1. Sign-aware feedback law

For a zero depth reference, use $u=K_p\delta z-K_d\delta v$. A diver who is too deep receives upward force; upward velocity is damped. Desired poles provide a direct design:

$$K_d=-m(p_1+p_2),\qquad K_p=m(p_1p_2-a).$$

In [ ]:
desired_poles = np.array([-0.6, -0.9])
K_d = -m*np.sum(desired_poles)
K_p = m*(np.prod(desired_poles)-a)
Acl = np.array([[0, -1], [a+K_p/m, -K_d/m]])
print(f"Kp={K_p:.2f} N/m, Kd={K_d:.2f} N s/m")
print("Closed-loop poles:", np.linalg.eigvals(Acl))
assert np.allclose(np.sort(np.linalg.eigvals(Acl)), np.sort(desired_poles))

## 2. Open-loop and closed-loop recovery

In [ ]:
def simulate_pd(x0, duration=20, disturbance=lambda t: 0.0, u_max=np.inf):
    def rhs(t, x):
        u_raw = K_p*x[0] - K_d*x[1]
        u = np.clip(u_raw, -u_max, u_max)
        return (A@x + B[:, 0]*(u + disturbance(t)))
    t_eval = np.linspace(0, duration, int(duration*50)+1)
    sol = solve_ivp(rhs, (0, duration), x0, t_eval=t_eval, rtol=1e-9, atol=1e-11)
    u_raw = K_p*sol.y[0]-K_d*sol.y[1]
    return sol.t, sol.y, np.clip(u_raw, -u_max, u_max)

t = np.linspace(0, 20, 1001)
open_sol = solve_ivp(lambda t,x: A@x, (0,20), [0.5,0], t_eval=t)
t, x_cl, u_cl = simulate_pd([0.5, 0])

fig, ax = plt.subplots()
ax.plot(t, open_sol.y[0], label="open loop")
ax.plot(t, x_cl[0], label="closed loop")
ax.set(xlabel="Time [s]", ylabel="Depth error [m]", title="Feedback stabilizes the unstable mode")
ax.legend(); plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8, 7))
axes[0].plot(t, x_cl[0]); axes[0].set_ylabel("Depth error [m]")
axes[1].plot(t, x_cl[1]); axes[1].set_ylabel("Upward velocity [m/s]")
axes[2].plot(t, u_cl); axes[2].set(xlabel="Time [s]", ylabel="Force [N]")
plt.show()

## 3. Constant disturbance and proportional offset

A constant unmodelled upward force produces the steady depth error

$$\delta z_{ss}=-\frac{d_0}{ma+K_p}.$$

PD feedback stabilizes the plant, but it does not guarantee zero offset.

In [ ]:
d0 = 5.0
disturbance = lambda t: d0 if t >= 5 else 0.0
t, x_d, u_d = simulate_pd([0,0], duration=35, disturbance=disturbance)
z_pred = -d0/(m*a+K_p)
print(f"Predicted offset {z_pred:.4f} m; simulated {x_d[0,-1]:.4f} m")

fig, ax = plt.subplots()
ax.plot(t, x_d[0])
ax.axhline(z_pred, color="k", ls=":", label="predicted steady offset")
ax.set(xlabel="Time [s]", ylabel="Depth error [m]", title="PD response to a constant disturbance")
ax.legend(); plt.show()

## 4. Actuator saturation

Pole placement describes the unsaturated linear model. A force limit changes the transient and can invalidate aggressive designs.

In [ ]:
t, x_free, _ = simulate_pd([2.0,0], u_max=np.inf)
t, x_sat, u_sat = simulate_pd([2.0,0], u_max=30.0)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8,6))
axes[0].plot(t, x_free[0], label="unlimited")
axes[0].plot(t, x_sat[0], label="±30 N")
axes[0].set_ylabel("Depth error [m]"); axes[0].legend()
axes[1].plot(t, u_sat); axes[1].axhline(30, color="k", ls=":"); axes[1].axhline(-30, color="k", ls=":")
axes[1].set(xlabel="Time [s]", ylabel="Applied force [N]")
plt.show()

## Engineering exercises

1. Choose poles $-0.3$ and $-1.2$. Compare gains and settling behavior.
2. Reduce the force limit until recovery from a 2 m error becomes unacceptable.
3. Change the disturbance sign and verify the offset formula.
4. Replace exact velocity with a noisy estimate and quantify the change in control effort.


In [ ]:
# Exercise starter: redesign the controller for a new pole pair
exercise_poles = np.array([-0.3, -1.2])
# Compute K_p and K_d, then verify the eigenvalues of A_cl.


## Summary

PD feedback moves the unstable plant poles into the left half-plane. The same experiment shows two limitations that the nominal pole calculation omits: constant disturbance offset and actuator saturation. Chapter 11 adds integral action and explicit anti-windup logic.